In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [8]:
class NCFDataset(Dataset):
    def __init__(self, user_ids, item_ids, ratings):
        self.user_ids = user_ids
        self.item_ids = item_ids
        self.ratings = ratings

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.user_ids[idx], self.item_ids[idx], self.ratings[idx]

In [9]:
# GMF Class (Generalized Matrix Factorization)
class GMF(nn.Module):
    def __init__(self, num_users, num_items, latent_dim):
        super(GMF, self).__init__()
        # Embedding layers for users and items
        self.user_embedding = nn.Embedding(num_users, latent_dim)
        self.item_embedding = nn.Embedding(num_items, latent_dim)

    def forward(self, user_ids, item_ids):
        user_emb = self.user_embedding(user_ids)  # Shape: [batch_size, latent_dim]
        item_emb = self.item_embedding(item_ids)  # Shape: [batch_size, latent_dim]
        
        # GMF: Element-wise product
        return user_emb * item_emb  # Shape: [batch_size, latent_dim]

In [10]:
# MLP Class (Multi-Layer Perceptron)
class MLP(nn.Module):
    def __init__(self, latent_dim, hidden_layers):
        super(MLP, self).__init__()
        input_dim = latent_dim * 2  # Concatenated user and item embeddings
        layers = []
        for units in hidden_layers:
            layers.append(nn.Linear(input_dim, units))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.2))
            input_dim = units
        self.mlp = nn.Sequential(*layers)

    def forward(self, user_ids, item_ids, user_embedding, item_embedding):
        # Concatenate user and item embeddings
        x = torch.cat([user_embedding, item_embedding], dim=-1)  # Shape: [batch_size, latent_dim * 2]
        return self.mlp(x)  # Shape: [batch_size, last_hidden_layer_size]

In [11]:
# NCF Class (Neural Collaborative Filtering) with GMF and MLP
class NCF(nn.Module):
    def __init__(self, num_users, num_items, latent_dim=16, hidden_layers=[64, 32, 16]):
        super(NCF, self).__init__()
        
        # GMF component
        self.gmf = GMF(num_users, num_items, latent_dim)
        
        # MLP component
        self.mlp = MLP(latent_dim, hidden_layers)
        
        # Output Layer (Fusion of GMF and MLP)
        fusion_dim = latent_dim + hidden_layers[-1]  # GMF (latent_dim) + MLP (last hidden layer)
        self.output_layer = nn.Linear(fusion_dim, 1)  # Single output value for regression
        
    def forward(self, user_ids, item_ids):
        # Get GMF output (element-wise product of embeddings)
        gmf_output = self.gmf(user_ids, item_ids)
        
        # Get MLP output (processed through hidden layers)
        user_emb = self.gmf.user_embedding(user_ids)
        item_emb = self.gmf.item_embedding(item_ids)
        mlp_output = self.mlp(user_ids, item_ids, user_emb, item_emb)
        
        # Combine GMF and MLP outputs
        combined = torch.cat([gmf_output, mlp_output], dim=-1)  # Shape: [batch_size, fusion_dim]
        
        # Final prediction (single value for regression)
        output = self.output_layer(combined)  # Shape: [batch_size, 1]
        return output

In [23]:
# Prepare sample data (change this with your actual dataset)
num_users = 1000
num_items = 500
user_ids = torch.randint(0, num_users, (10000,))
item_ids = torch.randint(0, num_items, (10000,))
ratings = torch.randint(0, 6, (10000,), dtype=torch.int)  # Ratings from 0 to 5

In [13]:
# Create dataset and dataloader
dataset = NCFDataset(user_ids, item_ids, ratings)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [19]:
# Initialize the model, loss function, and optimizer
model = NCF(num_users, num_items, latent_dim=16, hidden_layers=[64, 32, 16]).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [24]:
# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for user_ids, item_ids, ratings in train_loader:
        user_ids, item_ids, ratings = user_ids.to(device), item_ids.to(device), ratings.to(device)
        optimizer.zero_grad()
        predictions = model(user_ids, item_ids)
        loss = criterion(predictions.squeeze(), ratings)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

Epoch 1/10, Loss: 2.8880
Epoch 2/10, Loss: 2.8571
Epoch 3/10, Loss: 2.7897
Epoch 4/10, Loss: 2.7894
Epoch 5/10, Loss: 2.7650
Epoch 6/10, Loss: 2.7394
Epoch 7/10, Loss: 2.7050
Epoch 8/10, Loss: 2.6304
Epoch 9/10, Loss: 2.6014
Epoch 10/10, Loss: 2.5689


In [25]:
# Test: Making predictions
model.eval()
with torch.no_grad():
    test_user_ids = torch.tensor([0, 1, 2, 3]).to(device)  # Example test users
    test_item_ids = torch.tensor([10, 11, 12, 13]).to(device)  # Example test items
    predicted_ratings = model(test_user_ids, test_item_ids)
    print(predicted_ratings.squeeze())  # Predicted ratings for test users and itemss

tensor([2.5555, 3.4085, 2.4291, 2.0850], device='cuda:0')
